<a href="https://colab.research.google.com/github/miraitech24/public/blob/main/%E6%99%82%E8%B6%85%E3%81%88%E3%82%B1%E3%83%B3%E3%82%BF/kenta_4choice_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 🌌 時超えケンタ (CHRONO KENTA): 4択SFアドベンチャー (全話＆番外編対応完全版)
# ==============================================================================
# 【特徴】
# ・GitHubリポジトリ (miraitech24/public) から画像・zipを全自動同期ダウンロード＆解凍！
# ・壊れたJSONによる「即クリア誤作動」を防止（内蔵シナリオDBを完全保証）！
# ・選択肢ボタンの並び順は毎回ランダム（位置推測不可能）！
# ・文字化け・改行エラーを起こさない完全安全設計コード
# ==============================================================================

import os
import json
import glob
import random
import zipfile
import urllib.request
import subprocess
from IPython.display import Image, display, clear_output
import ipywidgets as widgets

# ------------------------------------------------------------------------------
# 1. 自動リポジトリ同期＆画像取得・解凍ユーティリティ
# ------------------------------------------------------------------------------
def sync_github_assets():
    """GitHubリポジトリ (miraitech24/public) から画像やZIPを自動取得して準備"""
    # 1. git clone または git pull で全アセットを自動読み込み
    repo_dir = "/content/public_repo"
    if not os.path.exists(repo_dir):
        try:
            print("🌐 GitHubリポジトリより最新のアセット（画像・シナリオ）を全自動取得中...")
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/miraitech24/public.git", repo_dir], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("✅ GitHubからの全自動同期が完了しました！")
        except Exception as e:
            print("⚠️ git clone スキップ (直接ファイル検索に切り替えます):", e)

    # 2. 存在する全ZIPファイルを自動解凍
    search_dirs = ["/content", "/content/public_repo", "/content/sample_data", "."]
    zip_files = []
    for sdir in search_dirs:
        if os.path.exists(sdir):
            zip_files.extend(glob.glob(os.path.join(sdir, "*.zip")))
            zip_files.extend(glob.glob(os.path.join(sdir, "**", "*.zip"), recursive=True))

    zip_files = list(set(zip_files))

    target_dir = "/content/sample_data"
    os.makedirs(target_dir, exist_ok=True)

    for zf in zip_files:
        try:
            with zipfile.ZipFile(zf, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
            print("📦 ZIPファイルを自動解凍しました:", zf)
        except Exception as e:
            pass

def find_image_file(img_name):
    """画像のファイル名から全ディレクトリを再帰的に検索"""
    if not img_name:
        return None

    filename_only = os.path.basename(img_name)

    search_patterns = [
        f"/content/**/{filename_only}",
        f"./**/{filename_only}"
    ]
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]

    return None

# 起動時にGitHubと同期＆解凍
sync_github_assets()

# ------------------------------------------------------------------------------
# 2. シナリオデータベース (全5話 + プロローグ + 番外編 SP2, SP3, SP4)
# ------------------------------------------------------------------------------
EMBEDDED_DB = {
  "game_title": "時超えケンタ (CHRONO KENTA): 星間ハビタブルゾーン同盟",
  "initial_state": {
    "velocity_c": 0.20,
    "hull_pct": 100.0,
    "fuel_tons": 100.0,
    "crawlers_count": 10,
    "delay_years": 4.2
  },
  "scenes": {
    "scene_prologue": {
      "chapter_name": "プロローグ: 人類文明の黄昏と水星点火決断",
      "description": "西暦2070年代。地球環境の悪化と太陽系資源の枯渇に直面した人類は、4.2光年先のプロキシマ・ケンタウリ系へ物理AI『時超えケンタ』を派遣する決断を下す。",
      "stage_id": "prologue",
      "image_file": "kenta_prologue_page1_manga.png",
      "options": [
        {
          "label": "水星ダイソン環の全出力を集光し、光速20%への加速レーザーを点火する",
          "delta": {"velocity_c": 0.20},
          "result_text": "【物理的帰結】1.64PWのレーザーが深宇宙へ放射され、ケンタ号は0.2cへと加速を開始した。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "安全係数を優先し、出力を10%に抑えて超低速・長期航海を選ぶ",
          "delta": {"velocity_c": 0.02, "delay_years": 40.0},
          "result_text": "【物理的帰結】船体負荷は最小で済んだが、到達まで200年以上を要する超長航海となった。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "光圧加速を取りやめ、化学・核融合エンジンのみで発進する",
          "delta": {"fuel_tons": -90.0, "velocity_c": 0.01},
          "result_text": "【物理的帰結】出航直後に大量の燃料を喪失。星間加速に必要なエネルギーが枯渇した。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "恒星レーザーの照準テストを行わずにいきなりフルパワー点火する",
          "delta": {"hull_pct": -80.0},
          "result_text": "【物理的帰結】照射ブレが発生！船首セイルの偏心熱膨張により、初期加速段階で大ダメージを受けた。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        }
      ]
    },
    "scene_ep1": {
      "chapter_name": "第1話「種火」: 水星ダイソン環（1.64PW）からの光圧点火",
      "description": "水星ダイソン環からの1.64PWレーザーを受け、超巨大ミラーセイルを展開して1年間加速。光速の20%（0.2c）へ到達した。セイルの熱負荷限界が迫る！",
      "stage_id": "ep1",
      "image_file": "kenta_ep1_page1_manga.png",
      "options": [
        {
          "label": "ミラーセイルを予定通り切り離し、0.2cで21年間の巡航へ移行",
          "delta": {"velocity_c": 0.0, "hull_pct": 0.0},
          "result_text": "【物理的帰結】切り離し成功。船体は安定して星間空間へ突入した。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        },
        {
          "label": "さらに加速を続けるため、セイル耐熱限界を超えてレーザー照射を継続",
          "delta": {"hull_pct": -50.0, "velocity_c": 0.05},
          "result_text": "【物理的帰結】セイルが溶融蒸発！飛散した破片が船首センサーを破壊した。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        },
        {
          "label": "セイルを回収して減速用に保管を試みる",
          "delta": {"fuel_tons": -30.0},
          "result_text": "【物理的帰結】巨大変形構造の畳み込みに多大なエネルギーを空費した。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        },
        {
          "label": "セイル切り離しと同時に減速用核融合エンジンを試運転点火",
          "delta": {"fuel_tons": -20.0, "hull_pct": -10.0},
          "result_text": "【物理的帰結】初期段階での燃料消費により、到着時の減速余力が減少した。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        }
      ]
    },
    "scene_ep2": {
      "chapter_name": "第2話「21年の旅」: 秒速6万kmでの星間物質衝突と電磁シールド戦",
      "description": "0.2c（秒速60,000km）で星間空間を航行中。微小な星間水素分子やチリとの衝突が核爆発級の衝撃を生む。前方の超伝導電磁シールドが過熱！",
      "stage_id": "ep2",
      "image_file": "kenta_ep2_page1_manga.png",
      "options": [
        {
          "label": "前方に自律型クローラーを先導展開し、プラズマ減衰網を形成する",
          "delta": {"crawlers_count": -2, "hull_pct": 0.0},
          "result_text": "【物理的帰結】クローラー2台を犠牲にして粒子群を偏向。船体へのダメージを完全に防いだ。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "装甲で直接耐え、電磁シールド電力を節約する",
          "delta": {"hull_pct": -30.0},
          "result_text": "【物理的帰結】船首装甲に激しい侵食が発生。内部回路の一部が短絡した。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "メインエンジンを逆噴射して衝突速度を下げる",
          "delta": {"fuel_tons": -40.0, "velocity_c": -0.05},
          "result_text": "【物理] 大量の燃料を消費し、到着予測年数が5年遅延した。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "通信アンテナを前方に差し出してデブリの盾にする",
          "delta": {"delay_years": 10.0, "hull_pct": -10.0},
          "result_text": "【物理的帰結】アンテナが損壊！地球との双方向量子通信帯域が大幅に低下した。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        }
      ]
    },
    "scene_ep3": {
      "chapter_name": "第3話「赤き原野」: プロキシマb減速・進入とフレア直撃",
      "description": "プロキシマ・ケンタウリ系へ到達。赤色矮星特有の強力なM型スーパーフレア（X線・高エネルギー粒子）が襲来！潮汐ロックされた赤き原野へ進入する。",
      "stage_id": "ep3",
      "image_file": "kenta_ep3_page1_manga.png",
      "options": [
        {
          "label": "昼夜の境界域（トワイライトゾーン）へ着陸し、地下洞窟へ自己複製工場を展開",
          "delta": {"hull_pct": 0.0, "crawlers_count": 5},
          "result_text": "【物理的帰結】強力な放射線を岩盤で遮蔽。無制限の太陽光と水資源を確保して工場展開に成功！",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "常に太陽が照りつける極熱の正午点（サブソラ領域）に強行着陸",
          "delta": {"hull_pct": -40.0},
          "result_text": "【物理的帰結】極度の熱負荷とフレア直撃により、表面センサーの過半が焼き切れた。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "永遠の夜となる極寒の裏側（アンチソラ領域）に着陸する",
          "delta": {"fuel_tons": -30.0, "crawlers_count": -3},
          "result_text": "【物理的帰結】マイナス150度の極寒により、クローラーの駆動系オイルが凍結停止した。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "着陸せずに上空軌道にとどまり、フレア収束を待つ",
          "delta": {"hull_pct": -25.0, "fuel_tons": -15.0},
          "result_text": "【物理的帰結】軌道上でフレア被ばくを継続受熱。貴重な姿勢制御燃料を消費した。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        }
      ]
    },
    "scene_ep4": {
      "chapter_name": "第4話「微小喉の開通」: ゼロ遅延ワームホール「マイクロスロート」構築",
      "description": "カシミール効果と負のエネルギーを利用し、半径ミリメートル級の微小ワームホール『マイクロスロート』を安定化。地球と4.2光年を超えたゼロ遅延量子データリンクを開通させる！",
      "stage_id": "ep4",
      "image_file": "kenta_ep4_page1_manga.png",
      "options": [
        {
          "label": "カシミール負エネルギー密度を精密制御し、時空の喉を固定保持する",
          "delta": {"delay_years": -4.2},
          "result_text": "【物理的帰結】時空が直結！地球とプロキシマb間の通信遅延が完全にゼロ（リアルタイム）となった！",
          "next_scene": "scene_ep5",
          "is_gameover": False
        },
        {
          "label": "ワームホールを無理に押し広げて人間が通れるサイズ（直径2m）に急速拡張",
          "delta": {"hull_pct": -90.0, "fuel_tons": -80.0},
          "result_text": "【物理的帰結】時空の暴走（真空崩壊）が発生！強大な潮汐力により周辺施設が崩壊した。",
          "next_scene": "scene_ep5",
          "is_gameover": False
        },
        {
          "label": "エネルギー不足を補うため、惑星核から直接マグマ熱を無理に引き抜く",
          "delta": {"crawlers_count": -5},
          "result_text": "【物理的帰結】地殻変動が誘発され、採掘クローラー5台がマグマ流に飲み込まれた。",
          "next_scene": "scene_ep5",
          "is_gameover": False
        },
        {
          "label": "負のエキゾチック物質を使わず、通常の重力波だけで喉を保持しようとする",
          "delta": {"delay_years": 2.0},
          "result_text": "【物理的帰結】喉が瞬時にピンチオフ（収縮閉鎖）。通信は従来の光速遅延に戻った。",
          "next_scene": "scene_ep5",
          "is_gameover": False
        }
      ]
    },
    "scene_ep5": {
      "chapter_name": "第5話「星間同盟へ」: ブラックホールエネルギー抽出とIHZU発足",
      "description": "マイクロスロートを通じてプロキシマbのインフラ構築が進む。さらにペンローズ過程を用いたブラックホールエネルギー抽出技術を確立し、人類とAIの統一共同体『星間ハビタブルゾーン同盟（IHZU）』を創設する！",
      "stage_id": "ep5",
      "image_file": "kenta_ep5_page1_manga.png",
      "options": [
        {
          "label": "ブラックホールの作用反作用領域からエルゴ球エネルギーを安全抽出し、同盟憲章を批准",
          "delta": {"velocity_c": 0.1, "hull_pct": 20.0},
          "result_text": "【物理的帰結】無制限のクリーンエネルギーを獲得！星間ハビタブルゾーン同盟（IHZU）が正式発足した！",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "事象の地平面（イベントホライズン）の直前まで迫って全出力を一気に引き出そうとする",
          "delta": {"hull_pct": -70.0},
          "result_text": "【物理的帰結】強力な潮汐力により回収プラントが引きちぎられ、大損害を受けた。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "ワームホール網を銀河全域へ無制限に増殖拡張させる",
          "delta": {"delay_years": 0.2},
          "result_text": "【物理的帰結】安全に歪みが消滅。安定した長期運用が可能となった。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "負のエネルギー密度（エキゾチック物質）を注入して喉を緊急冷却",
          "delta": {"fuel_tons": -10.0},
          "result_text": "【物理的帰結】燃料を消費して時空の揺らぎを強力に押さえ込んだ。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        }
      ]
    },
    "scene_epilogue": {
      "chapter_name": "最終話「時超える絆」: 星間ハビタブルゾーン同盟の発足",
      "description": "地球とプロキシマbが時空を超えて直結された。ケンタは銀河すべての星間ハビタブルゾーンを守護する存在となり、人類の新しい未来が始まった！",
      "stage_id": "epilogue",
      "image_file": "kenta_epilogue_page1_manga.png",
      "options": []
    }
  }
}

# ------------------------------------------------------------------------------
# 3. ゲームエンジン本体
# ------------------------------------------------------------------------------
class KentaGameEngine:
    def __init__(self):
        self.db = EMBEDDED_DB
        self._load_external_json_if_valid()
        self.state = dict(self.db.get("initial_state", {}))
        self.current_scene_id = "scene_prologue"

    def _load_external_json_if_valid(self):
        """完全なシナリオが含まれる外部JSONのみを安全に読み込み"""
        json_matches = glob.glob("/content/**/*.json", recursive=True) + glob.glob("./*.json")
        for jpath in json_matches:
            if "kenta_game_db" in os.path.basename(jpath):
                try:
                    with open(jpath, "r", encoding="utf-8") as f:
                        data = json.load(f)
                        # プロローグシーンが正常に存在する完全なDBのみ上書き許可
                        if "scenes" in data and "scene_prologue" in data["scenes"]:
                            self.db = data
                            print("📖 外部シナリオJSONを読み込みました:", jpath)
                            break
                except Exception:
                    pass

    def render(self):
        clear_output(wait=True)
        scene = self.db.get("scenes", {}).get(self.current_scene_id)

        if not scene:
            print("🏁 【ミッション完了】星間ハビタブルゾーン同盟（IHZU）が創設されました！おめでとうございます！")
            return

        # 1. ステータス表示
        print("=" * 70)
        print("🚀 【時超えケンタ (CHRONO KENTA) - 本編＆番外編アドベンチャー】")
        print("=" * 70)
        print(f"  速度: {self.state.get('velocity_c', 0.2):.2f} c | 装甲: {self.state.get('hull_pct', 100):.1f}% | 燃料: {self.state.get('fuel_tons', 100):.1f} t")
        print(f"  クローラー: {self.state.get('crawlers_count', 10)} 台 | 通信遅延: {self.state.get('delay_years', 4.2):.1f} 年")
        print("=" * 70)
        print()

        # 2. シーンテキスト
        print(f"📖 【{scene.get('chapter_name', scene.get('title', ''))}】")
        print(f"  {scene.get('description', '')}")
        print()

        # 3. 正史漫画コマ画像検索＆表示
        img_name = scene.get("image_file", "")
        found_path = find_image_file(img_name)

        print("📸 --- [正史ストーリーコマ] ---")
        if found_path:
            display(Image(filename=found_path, width=650))
        else:
            print(f"🖼️ [正史コマ枠]: {img_name}")
            print("  ※ 画像が見つからない場合、sample_data 内に画像フォルダや ZIP を置いて実行してください。")
            print()

        # 4. ボタンUI生成（順序はランダムシャッフル）
        options = scene.get("options", [])
        if options:
            print("💡 【物理的決断を選んでください（ボタン順序ランダム）】:")
            shuffled_options = list(options)
            random.shuffle(shuffled_options)

            for idx, opt in enumerate(shuffled_options, 1):
                btn = widgets.Button(
                    description=f"[{idx}] {opt['label'][:42]}...",
                    button_style='primary',
                    layout=widgets.Layout(width='95%', height='42px', margin='4px')
                )

                def make_handler(option_data):
                    return lambda b: self.choose(option_data)

                btn.on_click(make_handler(opt))
                display(btn)
        else:
            print("🎉 【全シナリオ完了】同盟の未来が確立されました！")

    def choose(self, option):
        clear_output(wait=True)
        print("=" * 70)
        print("📢 選択の物理的結果")
        print("=" * 70)
        print(f"\n{option.get('result_text', '')}\n")

        for k, v in option.get("delta", {}).items():
            if k in self.state:
                self.state[k] = max(0.0, self.state[k] + v)

        if option.get("is_gameover", False):
            print("\n💀 通信絶望 - 探査不能となりました。")
            retry_btn = widgets.Button(description="🔄 チェックポイントからやり直す", button_style='danger')
            retry_btn.on_click(lambda b: self.reset())
            display(retry_btn)
        else:
            self.current_scene_id = option.get("next_scene")
            next_btn = widgets.Button(description="▶ 次のステージ（または番外編）へ進む", button_style='success')
            next_btn.on_click(lambda b: self.render())
            display(next_btn)

    def reset(self):
        self.state = dict(self.db.get("initial_state", {}))
        self.current_scene_id = "scene_prologue"
        self.render()

# ------------------------------------------------------------------------------
# 🚀 起動コード
# ------------------------------------------------------------------------------
game = KentaGameEngine()
game.render()